In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [28]:
df = pd.read_csv('data_biomedical_sbl.csv')
print(df)

    Pulse_Rate  PTT  Systolic_BP
0           66  198        174.3
1           79  190        181.2
2           74  197        177.3
3           70  252        164.9
4           67  200        174.9
..         ...  ...          ...
95          72  206        174.7
96          68  227        167.5
97          74  213        175.8
98          72  258        163.6
99          60  259        159.0

[100 rows x 3 columns]


In [29]:
# Finding and Handling Duplicate Records
duplicates = df.duplicated()
print("\nDuplicate Records:\n", df[duplicates])

# observ: no duplicates


Duplicate Records:
 Empty DataFrame
Columns: [Pulse_Rate, PTT, Systolic_BP]
Index: []


In [31]:
# split dataset into train and test set. We would use training data to train the model

X = df[['Pulse_Rate','PTT']] # These are 2 indep variable. This is the feature that would predict the y
y = df['Systolic_BP']        # this is the response variable

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
# Lets see how many records are in training and testing dataset
print(f"X_train shape:{X_train.shape}")
print(f"X_test shape:{X_test.shape}")

X_train shape:(80, 2)
X_test shape:(20, 2)


## Lets check and deal with missing values

In [34]:
# check missing values
print(X_train.isnull().sum())
print(X_test.isnull().sum())

Pulse_Rate    0
PTT           0
dtype: int64
Pulse_Rate    0
PTT           0
dtype: int64


### Good. No missing values

In [35]:
# Modelling means picking up the right model: Here I choose LinearRegression
from sklearn.linear_model import LinearRegression

model = LinearRegression()

# Now I train the  model
model.fit(X_train, y_train)

LinearRegression()

In [43]:
# Lets make a prediction on SBP for person whose  Pulse_rate=66 and PTT=198 
X_values = [
    # pulse_rate, PTT
    [66, 198],  # patient1
]

new_data = pd.DataFrame(X_values, columns=X_train.columns)

SBP = model.predict(new_data)
print("The SBP is", SBP)

The SBP is [173.4075602]


# Predict values for larger dataset

In [37]:
X_values = [
    # pulse_rate, PTT
    [66, 198],  # patient1
    [70, 194],  # patient2
    [61,231],   # patient3
]

new_data = pd.DataFrame(X_values, columns=X_train.columns)

SBP = model.predict(new_data)
print("The SBP is", SBP)

The SBP is [173.4075602  176.22626035 164.23148786]


# Model Performance Evaluation

In [38]:
# STEP1: Make prediction on test data set
y_pred = model.predict(X_test)
print(y_pred)

[167.84241935 173.38079075 176.30616823 162.90586987 176.6109034
 178.42019335 161.59937292 167.83859515 168.95850689 173.4075602
 169.74691402 177.22419794 171.05723518 166.52827399 166.6234787
 173.50658911 164.83713399 176.32911348 180.85042625 168.25383185]


In [45]:
# step2: Compare predicted with actual value

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Compute errors
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

# Compute R-squared (R²) value
r2 = r2_score(y_test, y_pred)

# Print all results
print(f'Mean Absolute Error (MAE): {mae:.2f}')
print(f'Mean Squared Error (MSE) : {mse:.2f}')
print(f'R-squared                : {r2:.4f}')  # closer to 1 is good

Mean Absolute Error (MAE): 0.62
Mean Squared Error (MSE) : 0.59
R-squared                : 0.9796


### R-square value looking awesome

# Save the model
- After saving the model, you can deploy the model on your server or any app that you use.

In [40]:
# save the model
import joblib

joblib.dump(model, 'linear_regression_model.pkl')

['linear_regression_model.pkl']

# Load the model later

In [41]:
# load the model
import joblib

loaded_model = joblib.load('linear_regression_model.pkl')

In [42]:
# make prediction
X_values = [
    # pulse_rate, PTT
    [66, 198],  # patient1
    [70, 194],  # patient2
    [61,231],   # patient3
]

new_data = pd.DataFrame(X_values, columns=X_train.columns)

SBP = loaded_model.predict(new_data)
print(SBP)

[173.4075602  176.22626035 164.23148786]


## (OPTIONAL) Lets calculate SBP manually and also using the predict method

In [21]:
# Step1: Now lets extract a0, a1, a2:
a0 = model.intercept_ # intercept
a1 = model.coef_[0] # # x1-coefficient or slope
a2 = model.coef_[1] # # x2-coefficient or slope

print(f"The intercept a0: {a0}")
print(f"The x1-coeff a1: {a1}")
print(f"The x2-coeff a2: {a2}")


# step2: Lets write a simple code to calculate score based on eqn: SBP = a0 + a1 * pulse_rate + a2 * PTT

pulse_rate = 66
ptt        = 198

sbp = a0 + a1 * pulse_rate + a2 * ptt
print("calculated:", sbp)


# step3: Lets make a prediction on SBP for person whose  Pulse_rate=66 and PTT=198 
new_data = pd.DataFrame([[66, 198],], columns=X_train.columns)

SBP = model.predict(new_data)
print("predicted:", SBP)

### Both are same
# So I wanted to show that this predicted value can also be obtained via the line intercept and slope value

The intercept a0: 180.1958663738138
The x1-coeff a1: 0.502792998201693
The x2-coeff a2: -0.20188204069327462
calculated: 173.40756019785715
predicted: [173.4075602]
